<a href="https://colab.research.google.com/github/mjsheehan/lib_collections/blob/main/Subscription_Overlap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
WITH shaping_total_holdings AS(
SELECT
	subscription,
	collection,
	oclc_number,
	in_holdings,
	eissn,
	title,
	startdate,
	enddate,
	CoveragePeriod,
	embargoPeriod
FROM
	(SELECT *,
	COUNT(oclc_number) OVER(PARTITION BY oclc_number)-1 AS in_holdings,
	AGE(enddate, startdate) AS CoveragePeriod,
	AGE('2024-12-31', enddate) AS embargoPeriod FROM oclc_fulltx_fy25) X
),

ovelap_measure AS(
	SELECT
		subscription,
		collection,
		COUNT(oclc_number) AS titleCount,
		SUM(duplication) AS duplication
	FROM (SELECT *,
			CASE
				WHEN in_holdings = 0 THEN 0
				WHEN in_holdings > 0 THEN 1
			END AS duplication
		FROM shaping_total_holdings) Y
	GROUP BY 1,2)

SELECT
	collection,
	titleCount,
	duplication
	,COALESCE((NULLIF(duplication, 0) / titlecount::numeric)
    ::numeric(4,2),0.00) * 100 ||'%' AS overlap_perc
FROM overlap_measure
WHERE subscription = TRUE
ORDER BY (NULLIF(duplication, 0) / titlecount::numeric)::numeric(4,2) * 100 DESC;
